# Why is SparseGF2 fast for random measurements?

The circuits package is just orchestration; the *speed* lives in the
**core**, namely [`sparse_tableau.py`](../src/sparsegf2/core/sparse_tableau.py) and
its Numba kernels. This notebook explains, and **measures**, the one idea
that makes monitored (measurement-heavy) Clifford circuits cheap to simulate:
a Z-measurement is a **sparse rank-update**, not a dense $O(n^2)$ sweep.

## Contents
1. What a Z-measurement does to the tableau (Aaronson-Gottesman)
2. The dense cost: $O(n^2)$ per measurement, with a reference implementation
3. The sparse representation: why $Z_q$ only touches the generators with $X$ on $q$
4. Correctness: dense and sparse agree on the stabilizer subspace
5. Sparsity is real: stabilizer weight stays bounded under measurement
6. Operation-count head-to-head: dense $a\cdot 2n$ vs sparse $a\cdot w$
7. Wall-clock: full monitored-circuit runtime vs $n$
8. Why this matters for MIPT

**Sources.** Aaronson & Gottesman, *Improved Simulation of Stabilizer
Circuits*, [PRA 70, 052328 (2004)](https://arxiv.org/abs/quant-ph/0406196);
Fattal et al., [quant-ph/0406168](https://arxiv.org/abs/quant-ph/0406168).

## 1. What a Z-measurement does

A stabilizer state on $n$ qubits is tracked by $2n$ generators ($n$
*destabilizers* (rows $0..n{-}1$) and $n$ *stabilizers* (rows $n..2n{-}1$)),
each a phase-free Pauli stored as its $(x,z)$ bits in a $2n\times 2n$ binary
matrix $[X\,|\,Z]$. Measuring $Z_q$:

- A generator **anticommutes** with $Z_q$ iff its $X$-bit on qubit $q$ is set
  (symplectic product $= x_q$). Call the set of such rows the
  *anticommuters*, of size $a$.
- If no **stabilizer** anticommutes, the outcome is deterministic and the
  stabilizer subspace is unchanged.
- Otherwise pick one anticommuting stabilizer as the **pivot** $p$, XOR it
  into every *other* anticommuter (so they now commute with $Z_q$), demote
  the old pivot to a destabilizer, and install $Z_q$ as the new stabilizer.

The post-measurement **stabilizer subspace** is the unique projection onto
the $Z_q$ eigenspace, independent of the outcome *and* of which pivot you
pick. That invariance is what lets us cross-check two implementations.

In [1]:
import numpy as np
from sparsegf2 import SparseGF2, average_stabilizer_weight
from sparsegf2.core.symplectic import enumerate_sp4
from sparsegf2.core.linalg_gf2 import gf2_rref

TABLE = enumerate_sp4()          # 720 Sp(4,F2) matrices, native (no Stim)

def row_weight(M, r, n):
    '''Pauli weight of generator r: #qubits where it is non-identity.'''
    return int(np.count_nonzero(M[r, :n] | M[r, n:]))

print('table:', TABLE.shape)


table: (720, 4, 4)


## 2. The dense cost: $O(n^2)$ per measurement

The textbook dense CHP measurement does, per measurement:

- **scan** column $q$ of the $X$ block to find anticommuters, costing $O(n)$;
- **XOR** the pivot row into each of the $\le 2n$ other anticommuters, each
  XOR spanning the full width $2n$, which is $O(n)$ per row and $O(n^2)$ total.

Here it is, operating on the dense $[X|Z]$ matrix. The full-width row XORs
(`M[others, :] ^= M[p, :]`) are the $O(n^2)$ cost.

In [2]:
def dense_measure_z(M, q, n):
    '''Phase-free dense CHP Z-measurement on a (2n,2n) [X|Z] matrix.'''
    anti = np.nonzero(M[:, q])[0]              # x-bit at q -> anticommutes with Z_q
    stab_anti = anti[anti >= n]
    if stab_anti.size == 0:
        return M                               # deterministic: subspace unchanged
    p = int(stab_anti[0])                      # pivot = first anticommuting stabilizer
    others = anti[anti != p]
    M[others, :] ^= M[p, :]                    # <-- full-width XORs: O(a * 2n)
    M[p - n, :] = M[p, :]                      # old stabilizer -> destabilizer
    M[p, :] = 0
    M[p, n + q] = 1                            # new stabilizer = Z_q
    return M
print('dense reference defined')


dense reference defined


## 3. The sparse representation

SparseGF2 stores the *same* bits four complementary ways (see the
[`sparse_tableau` docstring](../src/sparsegf2/core/sparse_tableau.py)). Two
are the keys to a cheap measurement:

- a **per-generator support list**, the qubits where a generator is
  non-identity, so a row's nonzeros are iterated in $O(\text{weight})$, not
  $O(n)$;
- a **per-qubit inverted index of generators with the $X$-bit set**
  (`inv_x[q]`), so the anticommuters of $Z_q$ are found in $O(a)$ directly,
  with **no $O(n)$ column scan**.

So the sparse measurement does: look up the $a$ anticommuters in `inv_x[q]`
($O(a)$); pick the **minimum-weight** one as pivot (keeps the tableau
sparse); XOR it into the other $a{-}1$ anticommuters, each XOR touching only
the pivot's $w$ nonzeros. Total $\approx a\cdot w$, versus dense $a\cdot 2n$.
When generators are sparse ($w \ll n$), that is a large win.

## 4. Correctness: dense and sparse agree on the stabilizer subspace

Prepare a state, snapshot its $[X|Z]$, then measure the same qubit with both
implementations. The stabilizer blocks (rows $n..2n{-}1$) must have the same
**row span**, checked by comparing their GF(2) RREFs. (Destabilizers may
differ, because the pivots differ, but the physics is in the stabilizer subspace.)

In [3]:
def prepare(n, p, depth, seed):
    '''Nearest-neighbour brickwork + Z-measurements at rate p.'''
    gate_rng = np.random.default_rng(seed)
    sim = SparseGF2(n, rng=np.random.default_rng(seed + 1))
    for t in range(depth):
        for i in range(t % 2, n - 1, 2):                  # alternating NN pairs
            sim.apply_gate_2q(i, i + 1, TABLE[int(gate_rng.integers(0, 720))])
        if p > 0:
            for qq in np.nonzero(gate_rng.random(n) < p)[0]:
                sim.measure_z(int(qq))
    return sim

def stab_rref(M, n):
    return gf2_rref(np.ascontiguousarray(M[n:]))

n = 24
sim = prepare(n, p=0.1, depth=3 * n, seed=7)
M = sim.to_symplectic().copy()
q = n // 2
# sparse measure on the sim; dense measure on the snapshot
sim.measure_z(q)
dense_measure_z(M, q, n)
same = np.array_equal(stab_rref(sim.to_symplectic(), n), stab_rref(M, n))
print('dense and sparse give the SAME stabilizer subspace:', same)


dense and sparse give the SAME stabilizer subspace: True


## 5. Sparsity is real: weight stays bounded under measurement

The sparse cost $a\cdot w$ only beats $a\cdot 2n$ if the generator weight $w$
actually stays small. Measurements *keep it small*: each projection collapses
a qubit and prunes supports. Compare the **average stabilizer weight** in a
strongly-monitored circuit (area law, $p=0.2$) vs pure scrambling (volume
law, $p=0$), as $n$ grows. Under measurement the weight stays roughly flat;
without it, it grows with $n$.

In [4]:
print(f'{"n":>5} {"avg weight p=0.2":>18} {"avg weight p=0 (scrambled)":>28}')
for n in (16, 32, 48, 64):
    s_sparse = prepare(n, p=0.2, depth=4 * n, seed=1)
    s_dense = prepare(n, p=0.0, depth=4 * n, seed=1)
    print(f'{n:>5} {average_stabilizer_weight(s_sparse):>18.2f} '
          f'{average_stabilizer_weight(s_dense):>28.2f}')


    n   avg weight p=0.2   avg weight p=0 (scrambled)
   16               3.69                        12.38
   32               4.12                        24.19
   48               4.69                        35.77
   64               5.36                        47.88


## 6. Operation-count head-to-head

For one measurement on a prepared (monitored) state, count the real work:

- **dense**: $(a-1)\times 2n$, the non-pivot anticommuters, each XORed at
  full width.
- **sparse**: $(a-1)\times w_{\text{pivot}}$, the same anticommuters, each XOR
  touching only the (min-weight) pivot's nonzeros.

The ratio $\approx 2n / w_{\text{pivot}}$ grows with $n$ because $w$ stays
bounded under measurement. (SparseGF2 also avoids the $O(n)$ column scan via
`inv_x`, a further constant-factor win not even counted here.)

In [5]:
print(f'{"n":>5} {"a":>4} {"w_pivot":>8} {"dense=a*2n":>11} {"sparse=a*w":>11} {"ratio":>7}')
for n in (16, 32, 64, 128, 256, 512):
    sim = prepare(n, p=0.05, depth=8 * n, seed=3)
    M = sim.to_symplectic()
    q = n // 2
    anti = np.nonzero(M[:, q])[0]
    a = anti.size
    if a == 0:
        print(f'{n:>5} {0:>4}  (deterministic at this q/seed)')
        continue
    w_pivot = min(row_weight(M, int(r), n) for r in anti)   # min-weight pivot
    dense = (a - 1) * 2 * n
    sparse = (a - 1) * w_pivot
    ratio = dense / max(sparse, 1)
    print(f'{n:>5} {a:>4} {w_pivot:>8} {dense:>11} {sparse:>11} {ratio:>6.1f}x')


    n    a  w_pivot  dense=a*2n  sparse=a*w   ratio
   16   12        5         352          55    6.4x
   32   30       11        1856         319    5.8x
   64   38        2        4736          74   64.0x
  128   67        7       16896         462   36.6x
  256  128       10       65024        1270   51.2x
  512  244        4      248832         972  256.0x


## 7. Wall-clock: full monitored-circuit runtime vs $n$

End to end: time a complete $4n$-deep nearest-neighbour monitored circuit at
$p=0.2$. A dense CHP simulator costs $O(n)$ measurements $\times\,O(n^2)$ each
$= O(n^3)$ over the circuit (plus gates). SparseGF2 stays far below that
because both gates and measurements touch only sparse supports. (First call
includes a one-time Numba compile, excluded by a warmup.)

In [6]:
import time
prepare(8, 0.2, 8, 0)          # JIT warmup
print(f'{"n":>5} {"time_s (4n-deep, p=0.2)":>24} {"us / qubit-layer":>18}')
for n in (16, 32, 64, 128):
    depth = 4 * n
    t0 = time.perf_counter()
    prepare(n, p=0.2, depth=depth, seed=5)
    dt = time.perf_counter() - t0
    per = dt / (n * depth) * 1e6
    print(f'{n:>5} {dt:>24.4f} {per:>18.3f}')


    n  time_s (4n-deep, p=0.2)   us / qubit-layer
   16                   0.0031              3.069
   32                   0.0122              2.967
   64                   0.0415              2.535
  128                   0.1583              2.415


## 8. Why this matters for MIPT

Measurement-induced phase transition studies live in the **strongly
monitored** regime, exactly where this speedup is largest:

- High measurement rate $\Rightarrow$ **area-law** entanglement
  $\Rightarrow$ low stabilizer weight $\Rightarrow$ $w \ll n$ $\Rightarrow$
  the $2n/w$ advantage is at its biggest.
- The order parameters we record (half-cut entropy, code dimension,
  reference entropy) are all $\operatorname{rank}_{\mathbb{F}_2}$ quantities,
  computed on the same sparse tableau, no exponential state vector anywhere.

So the sparse representation is not a micro-optimization; it is what makes
large-$n$, deep, measurement-heavy circuits (the workhorse of MIPT) actually
tractable.

## Summary
- A Z-measurement is a **rank-update**: find the $a$ anticommuters, XOR a
  pivot into them.
- Dense pays $a\cdot 2n$ per measurement and $O(n)$ to find them; sparse pays
  $a\cdot w$ and finds them in $O(a)$ via `inv_x`.
- Measurements keep $w$ bounded, so the ratio $\sim 2n/w$ grows with $n$, as
  verified above on real tableaux.
- This is the engine under the circuits package; see
  [`notebooks/circuits/overview.ipynb`](circuits/overview.ipynb).